## CA1 - LUIS RAMIREZ ##

In [3]:
#Load pandas library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Creating a new CSV file from the Excel dataset for performance purposes.
## Only run the next cell once!

In [ ]:
## Changing the format file to a CSV
## Original path 
DATA_PATH = r"C:\Users\felip\Documents\CA-DataVisual\all_data_M_2023.xlsx"
excel_file = pd.read_excel(DATA_PATH)
## Path for the new CSV file
excel_file.to_csv(r"C:\Users\felip\Documents\CA-DataVisual\all_data_M_2023.csv", index=None, header=True)


## Running the new CSV file for this project

In [5]:
### Adding the new CSV DataFrame to df.
### With a new CSV file, the process will be faster.
csv_file = r"C:\Users\felip\Documents\CA-DataVisual\all_data_M_2023.csv"
df= pd.read_csv(csv_file)
df.head() #prove of the dataset working

C:\Users\felip\AppData\Local\Temp\ipykernel_40132\1920427162.py:4: DtypeWarning: Columns (4,13,14,15,16,30,31) have mixed types. Specify dtype option on import or set low_memory=False.
  df= pd.read_csv(csv_file)


,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,23.11,37.01,58.4,29050,35660,48060,76980,121470,NaN,NaN
1,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,56.19,81.29,111.36,54550,78330,116880,169090,231620,NaN,NaN
2,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,49.74,79.57,#,46400,66170,103460,165500,#,NaN,NaN
3,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,99.37,#,#,80000,130840,206680,#,#,NaN,NaN
4,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,99.37,#,#,80000,130840,206680,#,#,NaN,NaN


## Dealing with Missing Data ##

In [9]:
df.isnull().sum()

AREA                 0
AREA_TITLE           0
AREA_TYPE            0
PRIM_STATE           0
NAICS                0
NAICS_TITLE          0
I_GROUP              0
OWN_CODE             0
OCC_CODE             0
OCC_TITLE            0
O_GROUP              0
TOT_EMP          11598
EMP_PRSE             0
JOBS_1000       177501
LOC_QUOTIENT    177501
PCT_TOTAL       242857
PCT_RPT         242857
H_MEAN           20964
A_MEAN            5352
MEAN_PRSE            0
H_PCT10          20964
H_PCT25          21426
H_MEDIAN         23059
H_PCT75          26119
H_PCT90          33008
A_PCT10           5353
A_PCT25           5815
A_MEDIAN          7474
A_PCT75          10606
A_PCT90          17844
ANNUAL          396830
HOURLY          412572
dtype: int64

In [10]:
# Based on the information provided in the Excel file:
# '*'  = indicates that a wage estimate is not available
# '**'  = indicates that an employment estimate is not available
# '#'  = indicates a wage equal to or greater than $115.00 per hour or $239,200 per year
# '~' = indicates that the percent of establishments reporting the occupation is less than 0.5%

### STEP 1 ###
# Finding symbols to replace it
symbols = ['#', '*', '**', '~'] # Matching simbols.
df_str = df.astype(str).apply(lambda col: col.str.strip()) #  Turns every value in the table into text.
sym_table = pd.DataFrame({sym: (df_str == sym).sum() for sym in symbols}) # Compares every cell with one of the symbols
sym_table[(sym_table > 0).any(axis=1)] # Gives True if the symbols appear at least once.

,#,*,**,~
EMP_PRSE,0,0,11598,0
JOBS_1000,0,0,3633,0
LOC_QUOTIENT,0,0,3633,0
PCT_TOTAL,0,0,7886,0
PCT_RPT,0,0,7886,52694
MEAN_PRSE,0,4431,0,0


In [11]:
### STEP 2 ###
# Converting symbols to NaN (so they count as missing from now on)
num_cols = ['TOT_EMP','H_MEAN','A_MEAN','H_PCT10','H_PCT25','H_MEDIAN','H_PCT75','H_PCT90',
            'A_PCT10','A_PCT25','A_MEDIAN','A_PCT75','A_PCT90']
for c in num_cols:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(',', '').str.strip(), errors='coerce')

df[num_cols].isnull().sum()


### Run STEP 1 and STEP 2 again after this, and the counts will no include the symbols values.

TOT_EMP     11598
H_MEAN      20964
A_MEAN       5352
H_PCT10     20964
H_PCT25     21426
H_MEDIAN    23059
H_PCT75     26119
H_PCT90     33008
A_PCT10      5353
A_PCT25      5815
A_MEDIAN     7474
A_PCT75     10606
A_PCT90     17844
dtype: int64

# \### STEP 3 ###
## Choosing the AI, Data Science, Software Develpment occupations.
## The dataset has hundreds of occupations, so I keep only the tech ones (SOC 2018 codes), and I also keep
## two views of the data:
### ==> 'us': National totals (AREA_TYPE = 1), cross-industry
### ==> 'states' / 'metro' : state (AREA_TYPE = 2) and metro area (AREA_TYPE = 4) rows,, cross-industry.

In [13]:
tech_codes = {
    '15-1221': 'Computer & Info Research Scientists',
    '15-2051': 'Data Scientists',
    '15-1252': 'Software Developers',
    '15-1251': 'Computer Programmers',
    '15-1253': 'Software QA Analysts & Testers',
    '15-1211': 'Computer Systems Analysts',
    '15-1212': 'Information Security Analysts',
    '15-1242': 'Database Administrators',
    '15-1243': 'Database Architects',
    '15-1255': 'Web & Digital Interface Designers',
    '15-2031': 'Operations Research Analysts',
    '15-2041': 'Statisticians',
}
df['OCC_CODE'] = df['OCC_CODE'].astype(str).str.strip()
tech = df[
    df['OCC_CODE'].isin(tech_codes)
    & (df['NAICS_TITLE'] == 'Cross-industry')   # avoid double counting industries
].copy()
tech['OCC_SHORT'] = tech['OCC_CODE'].map(tech_codes)

us     = tech[tech['AREA_TYPE'] == 1].copy()
states = tech[tech['AREA_TYPE'] == 2].copy()
metro  = tech[tech['AREA_TYPE'] == 4].copy()

print(us.shape, states.shape, metro.shape)
us[['OCC_SHORT','TOT_EMP','A_MEDIAN']]

(12, 33) (596, 33) (2542, 33)


,OCC_SHORT,TOT_EMP,A_MEDIAN
135,Computer Systems Analysts,498810.0,103800.0
136,Information Security Analysts,175350.0,120360.0
138,Computer & Info Research Scientists,35210.0,145080.0
144,Database Administrators,76140.0,101510.0
145,Database Architects,59920.0,134700.0
148,Computer Programmers,120370.0,99700.0
149,Software Developers,1656880.0,132270.0
150,Software QA Analysts & Testers,203040.0,101800.0
152,Web & Digital Interface Designers,111060.0,98540.0
161,Operations Research Analysts,117880.0,83640.0


In [14]:
## Step 4 - Outliers